In [ ]:
# Cell 0: Configure PPO-style RoBERTa MLM LoRA experiments.
import os
import re
import ast
import json
import textwrap
import subprocess
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path('/home/rameyjm7/workspace/masked-emotion-lora-benchmark')
EXPRESS_ROOT = PROJECT_ROOT / 'express-emotion-recognition'
REGISTRY_PATH = PROJECT_ROOT / 'configs' / 'model_registry.yaml'
ENV_FILE = PROJECT_ROOT / 'env.txt'
ENV_PKL_FILE = PROJECT_ROOT / 'configs' / 'env.pkl'

MODEL_SEQUENCE = ['roberta_base', 'roberta_large']
ACTIVE_MODEL_INDEX = 0  # 0 = roberta_base first, then set to 1 for roberta_large.
MODEL_ID = MODEL_SEQUENCE[ACTIVE_MODEL_INDEX]

ACTIVE_EXPERIMENT = 0
EXPERIMENTS = [
    {
        'name': 'ppo_12k_qkv',
        'train_rows': 12000,
        'max_length': 512,
        'lr': 1.0e-5,
        'ppo_epochs': 2,
        'ppo_update_steps': 2,
        'train_batch_size': 4,
        'grad_accum': 1,
        'lora_r': 16,
        'lora_alpha': 32,
        'lora_dropout': 0.05,
        'target_modules': ['query', 'key', 'value'],
        'reward_exact': 1.0,
        'reward_vector_match': 0.5,
        'entropy_coef': 0.01,
        'sft_coef': 0.50,
        'clip_eps': 0.20,
        'seed': 42,
        'warmup_ratio': 0.06,
    },
    {
        'name': 'ppo_full_qkv_dense',
        'train_rows': None,
        'max_length': 512,
        'lr': 8.0e-6,
        'ppo_epochs': 3,
        'ppo_update_steps': 3,
        'train_batch_size': 2,
        'grad_accum': 2,
        'lora_r': 32,
        'lora_alpha': 64,
        'lora_dropout': 0.05,
        'target_modules': ['query', 'key', 'value', 'dense'],
        'reward_exact': 1.0,
        'reward_vector_match': 0.5,
        'entropy_coef': 0.01,
        'sft_coef': 0.40,
        'clip_eps': 0.20,
        'seed': 42,
        'warmup_ratio': 0.06,
    },
    {
        'name': 'ppo_full_qkv_dense_seed1337',
        'train_rows': None,
        'max_length': 512,
        'lr': 8.0e-6,
        'ppo_epochs': 3,
        'ppo_update_steps': 3,
        'train_batch_size': 2,
        'grad_accum': 2,
        'lora_r': 32,
        'lora_alpha': 64,
        'lora_dropout': 0.05,
        'target_modules': ['query', 'key', 'value', 'dense'],
        'reward_exact': 1.0,
        'reward_vector_match': 0.5,
        'entropy_coef': 0.01,
        'sft_coef': 0.40,
        'clip_eps': 0.20,
        'seed': 1337,
        'warmup_ratio': 0.06,
    },
]

assert 0 <= ACTIVE_EXPERIMENT < len(EXPERIMENTS), 'ACTIVE_EXPERIMENT out of range.'
EXP = EXPERIMENTS[ACTIVE_EXPERIMENT]
EXPERIMENT_NAME = EXP['name']
EXPERIMENT_MODEL_ID = f"{MODEL_ID}_{EXPERIMENT_NAME}"

RUN_TRAIN = True
RUN_INFERENCE = True
RUN_EVAL = True

TRAIN_ROWS = EXP['train_rows']
MAX_LENGTH = int(EXP['max_length'])
LR = float(EXP['lr'])
PPO_EPOCHS = int(EXP['ppo_epochs'])
PPO_UPDATE_STEPS = int(EXP['ppo_update_steps'])
TRAIN_BATCH_SIZE = int(EXP['train_batch_size'])
GRAD_ACCUM = int(EXP['grad_accum'])
LORA_R = int(EXP['lora_r'])
LORA_ALPHA = int(EXP['lora_alpha'])
LORA_DROPOUT = float(EXP['lora_dropout'])
TARGET_MODULES = list(EXP['target_modules'])
REWARD_EXACT = float(EXP['reward_exact'])
REWARD_VECTOR_MATCH = float(EXP['reward_vector_match'])
ENTROPY_COEF = float(EXP['entropy_coef'])
SFT_COEF = float(EXP['sft_coef'])
CLIP_EPS = float(EXP['clip_eps'])
SEED = int(EXP['seed'])
WARMUP_RATIO = float(EXP['warmup_ratio'])

# Queue controls: after base run, you can queue additional configs automatically.
RUN_EXPERIMENT_QUEUE = False
EXPERIMENT_QUEUE = [1, 2]
SKIP_COMPLETED_IN_QUEUE = True

REQUEST_DISTRIBUTED_TRAIN = False
TRAIN_GPU_ID = '0'
if TRAIN_GPU_ID is not None:
    os.environ['CUDA_VISIBLE_DEVICES'] = str(TRAIN_GPU_ID)

MODEL_CACHE_DIR = PROJECT_ROOT / 'outputs' / 'hf_cache' / f'hf_cache_{MODEL_ID}'
RUN_ROOT = PROJECT_ROOT / 'outputs' / 'lora_runs' / f'{MODEL_ID}_ppo_experiments'
RUN_DIR = RUN_ROOT / EXPERIMENT_NAME
METRICS_ROOT = PROJECT_ROOT / 'outputs' / 'metrics' / f'{MODEL_ID}_ppo_experiments'
METRICS_DIR = METRICS_ROOT / EXPERIMENT_NAME
FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures' / f'{MODEL_ID}_ppo_experiments'

EVAL_INPUT_PATH = METRICS_DIR / f'{EXPERIMENT_MODEL_ID}_baseline_eval_input.csv'
RAW_OUTPUT_PATH = METRICS_DIR / f'{EXPERIMENT_MODEL_ID}_ppo_raw.csv'
CLEAN_OUTPUT_PATH = METRICS_DIR / f'{EXPERIMENT_MODEL_ID}_ppo_clean.csv'
TRAIN_PREP_PATH = METRICS_DIR / f'{EXPERIMENT_MODEL_ID}_train_prep_stats.json'
METRICS_TABLE_PATH = PROJECT_ROOT / 'outputs' / 'metrics' / 'all_models_metrics.csv'
EXP_SUMMARY_PATH = METRICS_ROOT / 'experiment_summary.csv'
FIGURE_PATH = FIGURES_DIR / f'figure2_with_ppo_overlay_{EXPERIMENT_MODEL_ID}.png'

for p in [MODEL_CACHE_DIR, RUN_ROOT, RUN_DIR, METRICS_ROOT, METRICS_DIR, FIGURES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('MODEL_ID =', MODEL_ID)
print('ACTIVE_MODEL_INDEX =', ACTIVE_MODEL_INDEX)
print('EXPERIMENT_NAME =', EXPERIMENT_NAME)
print('EXPERIMENT_MODEL_ID =', EXPERIMENT_MODEL_ID)
print('RUN_DIR =', RUN_DIR)
print('METRICS_DIR =', METRICS_DIR)
print('CACHE_DIR =', MODEL_CACHE_DIR)
print('TARGET_MODULES =', TARGET_MODULES)
print('TRAIN_ROWS =', TRAIN_ROWS)
print('SEED =', SEED)
print('RUN_EXPERIMENT_QUEUE =', RUN_EXPERIMENT_QUEUE)
print('EXPERIMENT_QUEUE =', EXPERIMENT_QUEUE)



In [ ]:
# Cell 1: Load token, registry info, and command helpers.
import yaml
import pickle


def _existing_paths(paths):
    out = []
    seen = set()
    for p in paths:
        pp = Path(p).expanduser().resolve()
        if pp in seen:
            continue
        seen.add(pp)
        if pp.exists():
            out.append(pp)
    return out


def resolve_project_root() -> Path:
    candidates = []
    if 'PROJECT_ROOT' in globals():
        candidates.append(Path(PROJECT_ROOT))

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, cwd.parent, cwd.parent.parent])
    candidates.extend([
        Path('/home/rameyjm7/workspace/masked-emotion-lora-benchmark'),
        Path('/home/rameyjm7/workspace/llamafactory-gemma-lora'),
    ])

    for cand in _existing_paths(candidates):
        if (cand / 'configs' / 'model_registry.yaml').exists() and (cand / 'express-emotion-recognition').exists():
            return cand

    raise FileNotFoundError('Could not resolve project root containing configs/model_registry.yaml and express-emotion-recognition')


def load_hf_token(env_path: Path, pkl_path: Path) -> str:
    if pkl_path.exists():
        try:
            with pkl_path.open('rb') as f:
                payload = pickle.load(f)
            token = str(payload.get('HF_TOKEN', '')).strip()
            if token:
                return token
        except Exception as exc:
            print(f'Warning: failed to read {pkl_path}: {exc}')

    if env_path.exists():
        for raw in env_path.read_text(encoding='utf-8').splitlines():
            line = raw.strip()
            if not line or line.startswith('#'):
                continue
            if line.startswith('HF_TOKEN='):
                token = line.split('=', 1)[1].strip().strip('"').strip("'")
                if token:
                    return token

    raise ValueError(f'HF_TOKEN not found or empty in {pkl_path} or {env_path}')


def run_cmd(cmd, cwd=None, env=None, check=True):
    printable = ' '.join(str(x) for x in cmd)
    print(f'$ {printable}')
    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)

    proc = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        env=merged_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    lines = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        lines.append(line)

    rc = proc.wait()
    out = ''.join(lines)
    if check and rc != 0:
        tail = ''.join(lines[-120:])
        raise RuntimeError(
            f'Command failed ({rc}): {printable}\n'
            f'--- Last output lines ---\n{tail}'
        )
    return out


def parse_eval_metrics(stdout_text):
    metrics = {}
    for key in ['VRate', 'AccL', 'AccV', 'F1V', 'AccV-2']:
        m = re.search(rf'{re.escape(key)}\s*=\s*([0-9]*\.?[0-9]+)', stdout_text)
        if m:
            metrics[key] = float(m.group(1))
    return metrics


PROJECT_ROOT = resolve_project_root()
EXPRESS_ROOT = PROJECT_ROOT / 'express-emotion-recognition'
REGISTRY_PATH = PROJECT_ROOT / 'configs' / 'model_registry.yaml'
ENV_PKL_FILE = PROJECT_ROOT / 'configs' / 'env.pkl'
ENV_FILE = PROJECT_ROOT / 'env.txt'

MODEL_CACHE_DIR = PROJECT_ROOT / 'outputs' / 'hf_cache' / f'hf_cache_{MODEL_ID}'
RUN_ROOT = PROJECT_ROOT / 'outputs' / 'lora_runs' / f'{MODEL_ID}_ppo_experiments'
RUN_DIR = RUN_ROOT / EXPERIMENT_NAME
METRICS_ROOT = PROJECT_ROOT / 'outputs' / 'metrics' / f'{MODEL_ID}_ppo_experiments'
METRICS_DIR = METRICS_ROOT / EXPERIMENT_NAME
FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures' / f'{MODEL_ID}_ppo_experiments'

EVAL_INPUT_PATH = METRICS_DIR / f'{EXPERIMENT_MODEL_ID}_baseline_eval_input.csv'
RAW_OUTPUT_PATH = METRICS_DIR / f'{EXPERIMENT_MODEL_ID}_ppo_raw.csv'
CLEAN_OUTPUT_PATH = METRICS_DIR / f'{EXPERIMENT_MODEL_ID}_ppo_clean.csv'
TRAIN_PREP_PATH = METRICS_DIR / f'{EXPERIMENT_MODEL_ID}_train_prep_stats.json'
METRICS_TABLE_PATH = PROJECT_ROOT / 'outputs' / 'metrics' / 'all_models_metrics.csv'
EXP_SUMMARY_PATH = METRICS_ROOT / 'experiment_summary.csv'
FIGURE_PATH = FIGURES_DIR / f'figure2_with_ppo_overlay_{EXPERIMENT_MODEL_ID}.png'

for p in [MODEL_CACHE_DIR, RUN_ROOT, RUN_DIR, METRICS_ROOT, METRICS_DIR, FIGURES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

HF_TOKEN = load_hf_token(ENV_FILE, ENV_PKL_FILE)
os.environ['HF_TOKEN'] = HF_TOKEN

registry = yaml.safe_load(REGISTRY_PATH.read_text(encoding='utf-8'))['models']
registry_map = {m['id']: m for m in registry}
assert MODEL_ID in registry_map, f'{MODEL_ID} not found in {REGISTRY_PATH}'
MODEL_SPEC = registry_map[MODEL_ID]

MODEL_NAME_OR_PATH = MODEL_SPEC['model_name_or_path']
MODEL_SIZE_B = float(MODEL_SPEC['size_b'])

print('Resolved PROJECT_ROOT =', PROJECT_ROOT)
print('Loaded HF token using', ENV_PKL_FILE, 'with fallback to', ENV_FILE)
print('Model spec:', MODEL_SPEC)



In [ ]:
# Cell 2: Prepare train/eval datasets with isolated output paths.
import pandas as pd

train_source = EXPRESS_ROOT / 'data' / 'express.csv'
# Keep comparison anchored to original base eval file for apples-to-apples AccV.
eval_source = EXPRESS_ROOT / 'results' / 'roberta-base.csv'

train_df = pd.read_csv(train_source)
if TRAIN_ROWS is not None:
    train_df = train_df.head(int(TRAIN_ROWS))
train_df = train_df[['segment', 'labels', 'number_of_labels']].copy()

eval_df = pd.read_csv(eval_source)
eval_df = eval_df[['segment', 'labels', 'number_of_labels']].copy()
eval_df.to_csv(EVAL_INPUT_PATH, index=False)

print('Train rows:', len(train_df), 'from', train_source)
print('Eval rows:', len(eval_df), 'from', eval_source)
print('Saved eval input to', EVAL_INPUT_PATH)

display(train_df.head(2))
display(eval_df.head(2))



In [ ]:
# Cell 3: Train PPO-style RoBERTa MLM LoRA from notebook.
if RUN_TRAIN:
    train_script_path = RUN_DIR / 'train_roberta_mlm_ppo_worker.py'
    train_csv_path = EXPRESS_ROOT / 'data' / 'express.csv'
    lexicon_path = EXPRESS_ROOT / 'data' / 'lexicon-decomposition.csv'

    worker_script = textwrap.dedent("""
import os
import re
import ast
import json
import math
import random
import argparse
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
from torch.optim import AdamW
from torch.distributions import Categorical
from transformers import AutoTokenizer, AutoModelForMaskedLM, set_seed, get_scheduler
from peft import LoraConfig, get_peft_model


def safe_parse_label_list(raw):
    if isinstance(raw, list):
        return [str(x).strip().lower() for x in raw]
    if not isinstance(raw, str):
        return None
    txt = raw.strip()
    if not txt:
        return None
    try:
        val = ast.literal_eval(txt)
        if isinstance(val, list):
            return [str(x).strip().lower() for x in val]
    except Exception:
        return None
    return None


def normalize_word(text):
    return re.sub(r"\\s+", " ", str(text)).strip().lower()


def load_vec_map(path: str):
    df = pd.read_csv(path)
    out = {}
    for _, r in df.iterrows():
        w = normalize_word(r['word'])
        out[w] = tuple(int(v) for v in r.iloc[1:].tolist())
    return out


def build_examples(df, tokenizer, max_length):
    examples = []
    skipped_parse = 0
    skipped_mask_mismatch = 0
    skipped_multi_token = 0

    for row in df.itertuples(index=False):
        segment = str(row.segment)
        labels = safe_parse_label_list(row.labels)
        if labels is None:
            skipped_parse += 1
            continue

        enc = tokenizer(segment, truncation=True, max_length=max_length)
        input_ids = enc['input_ids']
        attention_mask = enc['attention_mask']
        mask_positions = [i for i, tok in enumerate(input_ids) if tok == tokenizer.mask_token_id]

        if len(mask_positions) != len(labels):
            skipped_mask_mismatch += 1
            continue

        target_ids = []
        target_words = []
        valid = True
        for lab in labels:
            lab = normalize_word(lab)
            ids = tokenizer(' ' + lab, add_special_tokens=False)['input_ids']
            if len(ids) != 1:
                ids = tokenizer(lab, add_special_tokens=False)['input_ids']
            if len(ids) != 1:
                valid = False
                break
            target_ids.append(int(ids[0]))
            target_words.append(lab)

        if not valid:
            skipped_multi_token += 1
            continue

        examples.append({
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'mask_positions': mask_positions,
            'target_ids': target_ids,
            'target_words': target_words,
        })

    stats = {
        'total_rows': int(len(df)),
        'kept_rows': int(len(examples)),
        'skipped_parse': int(skipped_parse),
        'skipped_mask_mismatch': int(skipped_mask_mismatch),
        'skipped_multi_token_labels': int(skipped_multi_token),
        'max_length': int(max_length),
    }
    return examples, stats


def pad_batch(examples, pad_token_id):
    max_len = max(len(x['input_ids']) for x in examples)
    input_rows = []
    attn_rows = []
    for ex in examples:
        pad_n = max_len - len(ex['input_ids'])
        input_rows.append(ex['input_ids'] + [pad_token_id] * pad_n)
        attn_rows.append(ex['attention_mask'] + [0] * pad_n)
    return (
        torch.tensor(input_rows, dtype=torch.long),
        torch.tensor(attn_rows, dtype=torch.long),
    )


def token_id_to_word(tok_id, tokenizer, cache):
    tid = int(tok_id)
    if tid not in cache:
        cache[tid] = normalize_word(tokenizer.decode([tid], skip_special_tokens=True))
    return cache[tid]


def build_rollout(old_logits, batch_examples, tokenizer, vec_map, reward_exact, reward_vector_match, decode_cache, device):
    batch_idx = []
    pos_idx = []
    action_ids = []
    old_logps = []
    target_ids = []
    rewards = []

    for i, ex in enumerate(batch_examples):
        positions = ex['mask_positions']
        t_ids = ex['target_ids']
        t_words = ex['target_words']

        for j, pos in enumerate(positions):
            logits_pos = old_logits[i, pos]
            dist = Categorical(logits=logits_pos)
            action = dist.sample()
            old_lp = dist.log_prob(action)

            pred_word = token_id_to_word(int(action.item()), tokenizer, decode_cache)
            target_word = normalize_word(t_words[j])

            if pred_word == target_word:
                r = reward_exact
            else:
                pred_vec = vec_map.get(pred_word)
                target_vec = vec_map.get(target_word)
                r = reward_vector_match if (pred_vec is not None and target_vec is not None and pred_vec == target_vec) else 0.0

            batch_idx.append(i)
            pos_idx.append(int(pos))
            action_ids.append(int(action.item()))
            old_logps.append(float(old_lp.item()))
            target_ids.append(int(t_ids[j]))
            rewards.append(float(r))

    if not rewards:
        return None

    rewards_t = torch.tensor(rewards, dtype=torch.float32, device=device)
    adv = rewards_t - rewards_t.mean()
    if float(adv.std().item()) > 1e-6:
        adv = adv / (adv.std() + 1e-6)

    return {
        'batch_idx': torch.tensor(batch_idx, dtype=torch.long, device=device),
        'pos_idx': torch.tensor(pos_idx, dtype=torch.long, device=device),
        'action_ids': torch.tensor(action_ids, dtype=torch.long, device=device),
        'target_ids': torch.tensor(target_ids, dtype=torch.long, device=device),
        'old_logps': torch.tensor(old_logps, dtype=torch.float32, device=device),
        'advantages': adv,
        'rewards': rewards_t,
    }


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--model_name_or_path', required=True)
    parser.add_argument('--train_source', required=True)
    parser.add_argument('--lexicon_path', required=True)
    parser.add_argument('--run_dir', required=True)
    parser.add_argument('--cache_dir', required=True)
    parser.add_argument('--train_prep_path', required=True)
    parser.add_argument('--train_rows', type=int, default=-1)
    parser.add_argument('--max_length', type=int, default=512)
    parser.add_argument('--learning_rate', type=float, default=1e-5)
    parser.add_argument('--ppo_epochs', type=int, default=2)
    parser.add_argument('--ppo_update_steps', type=int, default=2)
    parser.add_argument('--batch_size', type=int, default=4)
    parser.add_argument('--grad_accum', type=int, default=1)
    parser.add_argument('--model_id', required=True)
    parser.add_argument('--lora_r', type=int, default=16)
    parser.add_argument('--lora_alpha', type=int, default=32)
    parser.add_argument('--lora_dropout', type=float, default=0.05)
    parser.add_argument('--target_modules', type=str, default='query,key,value')
    parser.add_argument('--reward_exact', type=float, default=1.0)
    parser.add_argument('--reward_vector_match', type=float, default=0.5)
    parser.add_argument('--entropy_coef', type=float, default=0.01)
    parser.add_argument('--sft_coef', type=float, default=0.5)
    parser.add_argument('--clip_eps', type=float, default=0.2)
    parser.add_argument('--seed', type=int, default=42)
    parser.add_argument('--warmup_ratio', type=float, default=0.06)
    args = parser.parse_args()

    hf_token = os.environ.get('HF_TOKEN')
    if not hf_token:
        raise RuntimeError('HF_TOKEN is required in environment for training.')

    set_seed(int(args.seed))
    random.seed(int(args.seed))

    run_dir = Path(args.run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(
        args.model_name_or_path,
        token=hf_token,
        cache_dir=args.cache_dir,
        use_fast=True,
    )
    if tokenizer.mask_token_id is None:
        raise ValueError('Tokenizer has no mask token; cannot run MLM training.')
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token is not None else tokenizer.mask_token

    train_df = pd.read_csv(args.train_source)
    if args.train_rows is not None and args.train_rows >= 0:
        train_df = train_df.head(int(args.train_rows))
    train_df = train_df[['segment', 'labels', 'number_of_labels']].copy()

    examples, prep_stats = build_examples(train_df, tokenizer, int(args.max_length))
    Path(args.train_prep_path).write_text(json.dumps(prep_stats, indent=2), encoding='utf-8')
    print('Training prep stats:', prep_stats)

    if not examples:
        raise RuntimeError('No valid training examples were built.')

    vec_map = load_vec_map(args.lexicon_path)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    base_model = AutoModelForMaskedLM.from_pretrained(
        args.model_name_or_path,
        token=hf_token,
        cache_dir=args.cache_dir,
    )

    parsed_target_modules = [m.strip() for m in str(args.target_modules).split(',') if m.strip()]
    if not parsed_target_modules:
        parsed_target_modules = ['query', 'key', 'value']

    lora_cfg = LoraConfig(
        r=int(args.lora_r),
        lora_alpha=int(args.lora_alpha),
        lora_dropout=float(args.lora_dropout),
        bias='none',
        target_modules=parsed_target_modules,
    )
    model = get_peft_model(base_model, lora_cfg)
    model.to(device)
    model.train()
    model.print_trainable_parameters()

    optimizer = AdamW(model.parameters(), lr=float(args.learning_rate))

    steps_per_epoch = max(1, math.ceil(len(examples) / max(1, int(args.batch_size))))
    total_update_steps = max(1, steps_per_epoch * max(1, int(args.ppo_epochs)) * max(1, int(args.ppo_update_steps)))
    warmup_steps = int(float(args.warmup_ratio) * total_update_steps)
    scheduler = get_scheduler(
        name='linear',
        optimizer=optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_update_steps,
    )

    decode_cache = {}
    grad_accum = max(1, int(args.grad_accum))
    best_reward = -1e9
    best_epoch = None
    best_adapter_dir = run_dir / 'best_adapter'
    update_counter = 0

    history = []

    for epoch in range(int(args.ppo_epochs)):
        random.shuffle(examples)
        epoch_rewards = []
        epoch_policy = []
        epoch_ce = []
        epoch_entropy = []

        optimizer.zero_grad(set_to_none=True)

        for start in range(0, len(examples), int(args.batch_size)):
            batch_examples = examples[start:start + int(args.batch_size)]
            input_ids, attention_mask = pad_batch(batch_examples, tokenizer.pad_token_id)
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)

            with torch.no_grad():
                old_logits = model(input_ids=input_ids, attention_mask=attention_mask).logits

            rollout = build_rollout(
                old_logits,
                batch_examples,
                tokenizer,
                vec_map,
                float(args.reward_exact),
                float(args.reward_vector_match),
                decode_cache,
                device,
            )
            if rollout is None:
                continue

            epoch_rewards.extend([float(x) for x in rollout['rewards'].detach().cpu().tolist()])

            for _ in range(int(args.ppo_update_steps)):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                log_probs = torch.log_softmax(logits, dim=-1)
                probs = torch.softmax(logits, dim=-1)
                entropy_all = -(probs * log_probs).sum(dim=-1)

                b = rollout['batch_idx']
                p = rollout['pos_idx']
                a = rollout['action_ids']
                t = rollout['target_ids']

                new_logp = log_probs[b, p, a]
                old_logp = rollout['old_logps']
                adv = rollout['advantages']

                ratio = torch.exp(new_logp - old_logp)
                clipped = torch.clamp(ratio, 1.0 - float(args.clip_eps), 1.0 + float(args.clip_eps))
                policy_loss = -torch.min(ratio * adv, clipped * adv).mean()

                ce_loss = -log_probs[b, p, t].mean()
                entropy = entropy_all[b, p].mean()

                loss = policy_loss + float(args.sft_coef) * ce_loss - float(args.entropy_coef) * entropy
                (loss / grad_accum).backward()

                update_counter += 1
                if update_counter % grad_accum == 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True)

                epoch_policy.append(float(policy_loss.detach().cpu().item()))
                epoch_ce.append(float(ce_loss.detach().cpu().item()))
                epoch_entropy.append(float(entropy.detach().cpu().item()))

        if update_counter % grad_accum != 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        mean_reward = float(sum(epoch_rewards) / len(epoch_rewards)) if epoch_rewards else 0.0
        mean_policy = float(sum(epoch_policy) / len(epoch_policy)) if epoch_policy else 0.0
        mean_ce = float(sum(epoch_ce) / len(epoch_ce)) if epoch_ce else 0.0
        mean_entropy = float(sum(epoch_entropy) / len(epoch_entropy)) if epoch_entropy else 0.0

        if mean_reward > best_reward:
            best_reward = mean_reward
            best_epoch = int(epoch + 1)
            best_adapter_dir.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(str(best_adapter_dir))
            tokenizer.save_pretrained(str(best_adapter_dir))

        history.append({
            'epoch': int(epoch + 1),
            'mean_reward': mean_reward,
            'mean_policy_loss': mean_policy,
            'mean_ce_loss': mean_ce,
            'mean_entropy': mean_entropy,
        })

        print(
            f"epoch={epoch + 1} reward={mean_reward:.4f} policy={mean_policy:.4f} "
            f"ce={mean_ce:.4f} entropy={mean_entropy:.4f}"
        )

    model.save_pretrained(str(run_dir))
    tokenizer.save_pretrained(str(run_dir))

    summary = {
        'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
        'model_id': args.model_id,
        'model_name_or_path': args.model_name_or_path,
        'train_samples': int(len(examples)),
        'best_epoch': best_epoch,
        'best_reward': float(best_reward),
        'best_adapter_path': str(best_adapter_dir),
        'final_adapter_path': str(run_dir),
        'history': history,
        'ppo_epochs': int(args.ppo_epochs),
        'ppo_update_steps': int(args.ppo_update_steps),
        'batch_size': int(args.batch_size),
        'grad_accum': int(args.grad_accum),
        'clip_eps': float(args.clip_eps),
        'entropy_coef': float(args.entropy_coef),
        'sft_coef': float(args.sft_coef),
        'reward_exact': float(args.reward_exact),
        'reward_vector_match': float(args.reward_vector_match),
        'seed': int(args.seed),
    }
    (run_dir / 'training_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print('Saved PPO-style LoRA adapter to:', run_dir)
    print('Best adapter path:', best_adapter_dir)


if __name__ == '__main__':
    main()
    """).strip() + '\n'

    train_script_path.write_text(worker_script, encoding='utf-8')
    print('Wrote training worker script to:', train_script_path)

    train_rows_arg = -1 if TRAIN_ROWS is None else int(TRAIN_ROWS)

    common_args = [
        '--model_name_or_path', str(MODEL_NAME_OR_PATH),
        '--train_source', str(train_csv_path),
        '--lexicon_path', str(lexicon_path),
        '--run_dir', str(RUN_DIR),
        '--cache_dir', str(MODEL_CACHE_DIR),
        '--train_prep_path', str(TRAIN_PREP_PATH),
        '--train_rows', str(train_rows_arg),
        '--max_length', str(MAX_LENGTH),
        '--learning_rate', str(LR),
        '--ppo_epochs', str(PPO_EPOCHS),
        '--ppo_update_steps', str(PPO_UPDATE_STEPS),
        '--batch_size', str(TRAIN_BATCH_SIZE),
        '--grad_accum', str(GRAD_ACCUM),
        '--model_id', str(EXPERIMENT_MODEL_ID),
        '--lora_r', str(LORA_R),
        '--lora_alpha', str(LORA_ALPHA),
        '--lora_dropout', str(LORA_DROPOUT),
        '--target_modules', ','.join(TARGET_MODULES),
        '--reward_exact', str(REWARD_EXACT),
        '--reward_vector_match', str(REWARD_VECTOR_MATCH),
        '--entropy_coef', str(ENTROPY_COEF),
        '--sft_coef', str(SFT_COEF),
        '--clip_eps', str(CLIP_EPS),
        '--seed', str(SEED),
        '--warmup_ratio', str(WARMUP_RATIO),
    ]

    env = {
        'HF_TOKEN': HF_TOKEN,
        'TOKENIZERS_PARALLELISM': 'false',
    }

    _ = run_cmd(['python', str(train_script_path), *common_args], cwd=PROJECT_ROOT, env=env, check=True)
else:
    print('Training skipped (RUN_TRAIN=False).')



In [ ]:
# Cell 4: Run PPO-adapter RoBERTa inference on baseline-comparable rows.
if RUN_INFERENCE:
    import pandas as pd
    import torch
    from tqdm.auto import tqdm
    from transformers import AutoTokenizer, AutoModelForMaskedLM
    from peft import PeftModel

    def resolve_adapter_path(run_dir: Path) -> Path:
        summary_path = run_dir / 'training_summary.json'
        if summary_path.exists():
            try:
                obj = json.loads(summary_path.read_text(encoding='utf-8'))
                p = obj.get('best_adapter_path') or obj.get('final_adapter_path')
                if p:
                    pp = Path(str(p))
                    if pp.exists():
                        return pp
            except Exception as exc:
                print(f'Warning: failed to parse {summary_path}: {exc}')
        return run_dir

    eval_df = pd.read_csv(EVAL_INPUT_PATH)
    adapter_path = resolve_adapter_path(RUN_DIR)
    print('Using adapter path:', adapter_path)

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME_OR_PATH,
        token=HF_TOKEN,
        cache_dir=str(MODEL_CACHE_DIR),
        use_fast=True,
    )
    base_model = AutoModelForMaskedLM.from_pretrained(
        MODEL_NAME_OR_PATH,
        token=HF_TOKEN,
        cache_dir=str(MODEL_CACHE_DIR),
    )
    model = PeftModel.from_pretrained(base_model, str(adapter_path))

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device)
    model.eval()

    mask_token = tokenizer.mask_token
    if mask_token is None:
        raise ValueError('Tokenizer has no mask token; cannot run MLM inference.')

    def normalize_pred_token(text):
        txt = re.sub(r'\s+', ' ', str(text)).strip().lower()
        return txt

    def predict_mask_list(segment, num_labels):
        working_text = str(segment)
        target_n = int(num_labels)
        preds = []

        for _ in range(target_n):
            if mask_token not in working_text:
                break

            enc = tokenizer(
                working_text,
                return_tensors='pt',
                truncation=True,
                max_length=MAX_LENGTH,
            )
            enc = {k: v.to(device) for k, v in enc.items()}

            mask_positions = (enc['input_ids'][0] == tokenizer.mask_token_id).nonzero(as_tuple=False)
            if len(mask_positions) == 0:
                break

            pos = int(mask_positions[0].item())
            with torch.inference_mode():
                logits = model(**enc).logits[0, pos]
            pred_id = int(torch.argmax(logits).item())
            pred_word = normalize_pred_token(tokenizer.decode([pred_id], skip_special_tokens=True))
            if not pred_word:
                pred_word = 'neutral'

            preds.append(pred_word)
            working_text = working_text.replace(mask_token, pred_word, 1)

        if len(preds) != target_n:
            return 'INVALID OUTPUT'
        return str(preds)

    outputs = []
    for row in tqdm(eval_df.itertuples(index=False), total=len(eval_df), desc='RoBERTa PPO inference'):
        outputs.append(predict_mask_list(row.segment, row.number_of_labels))

    raw_df = eval_df.copy()
    raw_df['output'] = outputs
    raw_df.to_csv(RAW_OUTPUT_PATH, index=False)

    print('Saved raw outputs to:', RAW_OUTPUT_PATH)
    print('Rows:', len(raw_df))
else:
    print('Inference skipped (RUN_INFERENCE=False).')



In [ ]:
# Cell 5: Clean and evaluate outputs with existing project scripts.
lora_metrics = {}
if RUN_EVAL:
    _ = run_cmd(
        [
            'python', str(EXPRESS_ROOT / 'src' / 'evaluation' / 'result-cleaning.py'),
            '--input', str(RAW_OUTPUT_PATH),
            '--output', str(CLEAN_OUTPUT_PATH),
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )

    eval_stdout = run_cmd(
        [
            'python', str(EXPRESS_ROOT / 'src' / 'evaluation' / 'result-evaluation.py'),
            '--result_path', str(CLEAN_OUTPUT_PATH),
            '--breakdowns_path', str(EXPRESS_ROOT / 'data' / 'lexicon-decomposition.csv'),
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )

    lora_metrics = parse_eval_metrics(eval_stdout)
    print('PPO-style LoRA metrics:', lora_metrics)
else:
    print('Evaluation skipped (RUN_EVAL=False).')



In [ ]:
# Cell 6: Upsert shared metrics table and experiment summary.
import pandas as pd

row = {
    'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
    'model_id': EXPERIMENT_MODEL_ID,
    'family': MODEL_SPEC['family'],
    'model_name_or_path': MODEL_NAME_OR_PATH,
    'size_b': MODEL_SIZE_B,
    'pipeline': 'hf_peft_mlm_ppo_style',
    'is_lora': True,
    'v_rate': lora_metrics.get('VRate'),
    'accl': lora_metrics.get('AccL'),
    'accv': lora_metrics.get('AccV'),
    'f1v': lora_metrics.get('F1V'),
    'accv2': lora_metrics.get('AccV-2'),
    'clean_output_path': str(CLEAN_OUTPUT_PATH),
    'run_dir': str(RUN_DIR),
}

if METRICS_TABLE_PATH.exists():
    metrics_df = pd.read_csv(METRICS_TABLE_PATH)
else:
    metrics_df = pd.DataFrame()

if not metrics_df.empty and {'model_id', 'is_lora'}.issubset(metrics_df.columns):
    metrics_df = metrics_df[~((metrics_df['model_id'] == EXPERIMENT_MODEL_ID) & (metrics_df['is_lora'].astype(str).str.lower() == 'true'))]

metrics_df = pd.concat([metrics_df, pd.DataFrame([row])], ignore_index=True)
metrics_df.to_csv(METRICS_TABLE_PATH, index=False)
print('Updated metrics table:', METRICS_TABLE_PATH)

display(metrics_df.tail(12))

summary_row = {
    'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
    'base_model_id': MODEL_ID,
    'experiment_model_id': EXPERIMENT_MODEL_ID,
    'experiment_name': EXPERIMENT_NAME,
    'train_rows': TRAIN_ROWS,
    'max_length': MAX_LENGTH,
    'lr': LR,
    'ppo_epochs': PPO_EPOCHS,
    'ppo_update_steps': PPO_UPDATE_STEPS,
    'train_batch_size': TRAIN_BATCH_SIZE,
    'grad_accum': GRAD_ACCUM,
    'lora_r': LORA_R,
    'lora_alpha': LORA_ALPHA,
    'lora_dropout': LORA_DROPOUT,
    'target_modules': ','.join(TARGET_MODULES),
    'reward_exact': REWARD_EXACT,
    'reward_vector_match': REWARD_VECTOR_MATCH,
    'entropy_coef': ENTROPY_COEF,
    'sft_coef': SFT_COEF,
    'clip_eps': CLIP_EPS,
    'seed': SEED,
    'v_rate': lora_metrics.get('VRate'),
    'accl': lora_metrics.get('AccL'),
    'accv': lora_metrics.get('AccV'),
    'f1v': lora_metrics.get('F1V'),
    'accv2': lora_metrics.get('AccV-2'),
    'clean_output_path': str(CLEAN_OUTPUT_PATH),
    'run_dir': str(RUN_DIR),
}

if EXP_SUMMARY_PATH.exists():
    exp_df = pd.read_csv(EXP_SUMMARY_PATH)
else:
    exp_df = pd.DataFrame()

if not exp_df.empty and {'experiment_model_id'}.issubset(exp_df.columns):
    exp_df = exp_df[exp_df['experiment_model_id'] != EXPERIMENT_MODEL_ID]

exp_df = pd.concat([exp_df, pd.DataFrame([summary_row])], ignore_index=True)
exp_df.to_csv(EXP_SUMMARY_PATH, index=False)
print('Updated experiment summary:', EXP_SUMMARY_PATH)
display(exp_df.tail(20))



In [ ]:
# Cell 7: Quick compare table for this model's PPO-style runs.
import pandas as pd

if EXP_SUMMARY_PATH.exists():
    df = pd.read_csv(EXP_SUMMARY_PATH)
    show_cols = [
        'experiment_name', 'accv', 'accl', 'f1v',
        'train_rows', 'lr', 'ppo_epochs', 'ppo_update_steps',
        'train_batch_size', 'grad_accum', 'target_modules', 'seed'
    ]
    show_cols = [c for c in show_cols if c in df.columns]
    out = df.sort_values('accv', ascending=False)[show_cols]
    display(out)
else:
    print('No experiment summary found yet:', EXP_SUMMARY_PATH)

print('Next step: set ACTIVE_MODEL_INDEX = 1 in Cell 0 to run roberta_large with the same notebook.')

